# Spatial LR NMF factors across histological regions

This notebook contains the analysis used to inspect and quantify the spatial
distribution of NMF-derived ligand–receptor (LR) factors across histological
regions in the Visium HD tissue.

Only the workflow supporting the manuscript regional-factor analysis is retained:

1. load the spatial LIANA+ LR object with histological-region annotations;
2. retain spatially significant LR pairs;
3. perform four-factor non-negative matrix factorization (NMF);
4. visualize factor activity across the tissue;
5. summarize factor activity across histological regions; and
6. generate the regional enrichment heatmap and factor-activity violin plots.

Repository-relative paths are used in place of local-computer paths.


In [ ]:
from pathlib import Path

import liana as li
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc


## Load spatial LR data

The input AnnData object is the output of the LIANA+ spatial LR analysis and is
expected to contain:

- LR variables with `ligand`, `receptor`, `morans`, and `morans_pvals` in `.var`;
- histological-region labels in `obs["Region"]`; and
- spatial coordinates in `obsm["spatial"]`.


In [ ]:
DATA_PATH = Path("data/spatial_lr/lrdata_region.h5ad")

lrdata = sc.read_h5ad(DATA_PATH)

required_var = {"ligand", "receptor", "morans", "morans_pvals"}
missing_var = required_var.difference(lrdata.var.columns)
if missing_var:
    raise KeyError(f"Missing required var columns: {sorted(missing_var)}")

if "Region" not in lrdata.obs.columns:
    raise KeyError("lrdata.obs['Region'] is required")

if "spatial" not in lrdata.obsm:
    raise KeyError("lrdata.obsm['spatial'] is required")

print(lrdata)
print("\nHistological regions:")
print(lrdata.obs["Region"].value_counts())
print("\nTotal LR pairs:", lrdata.n_vars)


## Retain spatially significant LR pairs

LR pairs with positive bivariate Moran's I and empirical P < 0.05 are retained
for NMF, matching the manuscript analysis.


In [ ]:
sig_mask = (
    (lrdata.var["morans"] > 0)
    & (lrdata.var["morans_pvals"] < 0.05)
)

lrdata_nmf = lrdata[:, sig_mask].copy()

print("Total evaluated LR pairs:", lrdata.n_vars)
print("Retained spatial LR pairs:", lrdata_nmf.n_vars)


## Four-factor NMF

The retained spatial LR signals are decomposed into four factors using LIANA+.
`NMF_W` contains the spatial factor scores for individual Visium HD bins,
whereas `NMF_H` contains LR-pair loadings for each factor.


In [ ]:
li.multi.nmf(
    lrdata_nmf,
    n_components=4,
    inplace=True,
    random_state=0,
    max_iter=500,
    verbose=True,
)

factor_scores = li.ut.get_factor_scores(
    lrdata_nmf,
    obsm_key="NMF_W",
)

lr_loadings = (
    li.ut.get_variable_loadings(
        lrdata_nmf,
        varm_key="NMF_H",
    )
    .set_index("index")
)

factor_cols = list(lr_loadings.columns)

print("Spatial factor matrix:", lrdata_nmf.obsm["NMF_W"].shape)
print("LR loading matrix:", lrdata_nmf.varm["NMF_H"].shape)
print("Factors:", factor_cols)


## Spatial distribution of NMF factors

A lightweight AnnData object is constructed from the spatial factor-score matrix
for visualization in tissue coordinates.


In [ ]:
nmf = sc.AnnData(
    X=lrdata_nmf.obsm["NMF_W"].copy(),
    obs=lrdata_nmf.obs.copy(),
    var=pd.DataFrame(index=factor_cols),
)

nmf.obsm["spatial"] = lrdata_nmf.obsm["spatial"].copy()

axs = sc.pl.embedding(
    nmf,
    basis="spatial",
    color=factor_cols,
    cmap="coolwarm",
    vmin=0,
    vmax=0.05,
    size=10,
    ncols=2,
    show=False,
)

for ax in np.ravel(axs):
    ax.invert_yaxis()
    ax.set_aspect("equal")

plt.show()


## Quantify factor activity across histological regions

Regional mean and median factor scores are calculated directly from the NMF
spatial factor scores.


In [ ]:
factor_df = pd.DataFrame(
    nmf.X,
    index=nmf.obs_names,
    columns=factor_cols,
)

factor_df["Region"] = nmf.obs["Region"].astype(str).values

region_order = [
    region
    for region in ["Alveolar", "Gran", "Stromal", "TLS"]
    if region in factor_df["Region"].unique()
]

region_factor_mean = (
    factor_df
    .groupby("Region", observed=True)[factor_cols]
    .mean()
    .reindex(region_order)
)

region_factor_median = (
    factor_df
    .groupby("Region", observed=True)[factor_cols]
    .median()
    .reindex(region_order)
)

print("Mean factor activity:")
display(region_factor_mean)

print("Median factor activity:")
display(region_factor_median)


## Regional enrichment heatmap

Each NMF factor is z-scored across all spatial bins and the mean normalized
activity is then calculated within each histological region. This produces the
regional-enrichment heatmap shown in Extended Data Fig. 7.


In [ ]:
factor_z = factor_df[factor_cols].copy()

factor_z = (
    factor_z - factor_z.mean(axis=0)
) / factor_z.std(axis=0)

factor_z["Region"] = factor_df["Region"].values

region_factor_z = (
    factor_z
    .groupby("Region", observed=True)[factor_cols]
    .mean()
    .reindex(region_order)
)

region_display = {
    "Alveolar": "Alveolar",
    "Gran": "Granuloma",
    "Stromal": "Stromal",
    "TLS": "TLS",
}

heatmap_df = region_factor_z.rename(index=region_display)

fig, ax = plt.subplots(
    figsize=(5, 4),
    dpi=300,
)

im = ax.imshow(
    heatmap_df.values,
    aspect="auto",
    cmap="RdBu_r",
)

ax.set_xticks(
    np.arange(len(factor_cols))
)
ax.set_xticklabels(
    [str(i + 1) for i in range(len(factor_cols))]
)

ax.set_yticks(
    np.arange(len(heatmap_df.index))
)
ax.set_yticklabels(
    heatmap_df.index
)

ax.set_xlabel("NMF factor")
ax.set_ylabel("Region")
ax.set_title("Regional enrichment of spatial LR factors")

cbar = fig.colorbar(
    im,
    ax=ax,
)
cbar.set_label("Normalized factor activity")

plt.tight_layout()
plt.show()


## Factor activity distributions by region

The manuscript displays regional activity distributions for Factors 1–3.
Histological regions are abbreviated as:

- AL, alveolar;
- MY, granuloma/myeloid;
- SI, stromal interface;
- LA, lymphoid aggregate/TLS.

Factor 4 remains included in the regional enrichment heatmap above.


In [ ]:
region_abbrev = {
    "Alveolar": "AL",
    "Gran": "MY",
    "Stromal": "SI",
    "TLS": "LA",
}

nmf.obs["Region_plot"] = (
    nmf.obs["Region"]
    .astype(str)
    .map(region_abbrev)
)

region_plot_order = [
    region_abbrev[region]
    for region in region_order
]

nmf.obs["Region_plot"] = pd.Categorical(
    nmf.obs["Region_plot"],
    categories=region_plot_order,
    ordered=True,
)

violin_factors = [
    factor
    for factor in ["Factor1", "Factor2", "Factor3"]
    if factor in factor_cols
]

sc.pl.violin(
    nmf,
    keys=violin_factors,
    groupby="Region_plot",
    stripplot=False,
    multi_panel=True,
    rotation=0,
)


## Optional export of regional summaries

These tables contain the values used to inspect regional factor activity.
Uncomment the lines below if tabular output is required.


In [ ]:
# OUTPUT_DIR = Path("output")
# OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
#
# region_factor_mean.to_csv(
#     OUTPUT_DIR / "NMF_factor_activity_region_mean.csv"
# )
#
# region_factor_median.to_csv(
#     OUTPUT_DIR / "NMF_factor_activity_region_median.csv"
# )
#
# region_factor_z.to_csv(
#     OUTPUT_DIR / "NMF_factor_activity_region_normalized.csv"
# )
